# GBC063 · Semana 1 · Notebook — O que é Inteligência Artificial?

**Prof. Marcelo Keese Albertini · FACOM · UFU**

Este notebook acompanha o [material de estudo](../semana01/material_introducao_ia.html).
Execute cada célula, modifique os parâmetros, quebre os agentes — e entenda.

## 1. O agente mais simples do mundo: um termostato

Percepção: temperatura atual. Decisão: ligar/desligar/dispositivo. Ação: comando.

In [ ]:
def agente_termostato(temperatura_atual, alvo=22):
    """Agente reativo simples: decide baseado apenas na percepção atual."""
    if temperatura_atual < alvo:
        return "LIGAR_AQUECEDOR"
    elif temperatura_atual > alvo:
        return "LIGAR_AR_CONDICIONADO"
    return "DESLIGAR"

# Teste: simule um dia de variação de temperatura
temperaturas = [18, 19, 20, 21, 22, 23, 24, 25, 24, 22]
for t in temperaturas:
    acao = agente_termostato(t)
    print(f"{t}°C → {acao}")

### 🧪 Experimento: Quebre o termostato

1. O que acontece se o alvo for exatamente a temperatura atual? O agente decide corretamente?
2. Modifique o agente para que ele tenha histerese: só liga o aquecedor se `temp < alvo - 1` e só liga o ar se `temp > alvo + 1`.
3. Por que histerese é importante em sistemas reais?

In [ ]:
# Seu experimento aqui
def agente_termostato_histerese(temp, alvo=22, margem=1):
    # Implemente a histerese
    pass

## 2. O agente aspirador de pó

O ambiente é uma linha de 2 cômodos (A e B). O agente percebe sua posição e se o cômodo atual está sujo.

In [ ]:
def agente_aspirador(percepcao):
    """Agente reativo para um mundo de 2 cômodos em linha.
    
    Args:
        percepcao: tupla (posicao, sujo) — ex: ("A", True)
    Returns:
        ação: "ASPIRAR", "DIREITA", ou "ESQUERDA"
    """
    posicao, sujo = percepcao
    if sujo:
        return "ASPIRAR"       # ← a linha mais importante
    if posicao == "A":
        return "DIREITA"
    return "ESQUERDA"

# Simulação completa
def simular_aspirador(estado_inicial, max_passos=20):
    """Simula o agente aspirador em um ambiente de 2 cômodos."""
    comodos = list(estado_inicial)  # [sujo_A, sujo_B]
    pos = "A"
    historico = []
    
    for passo in range(max_passos):
        idx = 0 if pos == "A" else 1
        percepcao = (pos, comodos[idx])
        acao = agente_aspirador(percepcao)
        historico.append((passo, pos, comodos.copy(), acao))
        
        if acao == "ASPIRAR":
            comodos[idx] = False
        elif acao == "DIREITA" and pos == "A":
            pos = "B"
        elif acao == "ESQUERDA" and pos == "B":
            pos = "A"
        
        if not any(comodos):
            print(f"✓ Ambiente limpo em {passo + 1} passos!")
            break
    
    return historico

# Teste: ambos os cômodos sujos
print("=== Teste: Ambos sujos ===")
hist = simular_aspirador([True, True])
for p, pos, comodos, acao in hist:
    print(f"  Passo {p}: pos={pos}, cômodos={comodos} → {acao}")

print("\n=== Teste: Apenas A sujo ===")
hist = simular_aspirador([True, False])
for p, pos, comodos, acao in hist:
    print(f"  Passo {p}: pos={pos}, cômodos={comodos} → {acao}")

### 🧪 Experimento: Generalize o aspirador

1. O agente funciona com **3 cômodos**? Modifique o código e descubra.
2. Adicione **ruído no sensor**: 20% de chance do sensor reportar o estado errado. O agente ainda limpa tudo?
3. Como você modificaria o agente para que ele **lembrasse** quais cômodos já visitou?

In [ ]:
# Seu experimento com 3 cômodos e/ou ruído
import random

def agente_aspirador_3comodos(percepcao):
    """Adapte o agente para 3 cômodos (A, B, C).
    
    Dica: pense no que acontece quando o agente está em B ou C.
    """
    posicao, sujo = percepcao
    # Seu código aqui
    pass

def simular_3comodos(estado_inicial, ruido=0.0, max_passos=30):
    """Simulação com 3 cômodos e ruído de sensor opcional."""
    comodos = list(estado_inicial)  # [A, B, C]
    pos = "B"  # começa no meio
    
    for passo in range(max_passos):
        idx = {"A": 0, "B": 1, "C": 2}[pos]
        sensor = comodos[idx]
        if random.random() < ruido:
            sensor = not sensor  # ruído: inverte a leitura
        
        percepcao = (pos, sensor)
        acao = agente_aspirador_3comodos(percepcao)
        print(f"  Passo {passo}: pos={pos}, real={'sujo' if comodos[idx] else 'limpo'}, sensor={'sujo' if sensor else 'limpo'} → {acao}")
        
        if acao == "ASPIRAR":
            comodos[idx] = False
        elif acao == "DIREITA" and pos in ("A", "B"):
            pos = {"A": "B", "B": "C"}[pos]
        elif acao == "ESQUERDA" and pos in ("B", "C"):
            pos = {"B": "A", "C": "B"}[pos]
        
        if not any(comodos):
            print(f"  ✓ Ambiente limpo em {passo + 1} passos!")
            break
    else:
        print(f"  ⚠️ Não limpou tudo em {max_passos} passos. Cômodos: {comodos}")

# Teste com 3 cômodos
print("=== 3 cômodos, A sujo, sem ruído ===")
simular_3comodos([True, False, False], ruido=0.0)

## 3. O loop universal da IA

Esta é a interface que usaremos em **todas as semanas do curso**, do termostato ao AlphaGo:

In [ ]:
# O loop universal — memorize esta estrutura
def loop_agente(ambiente, agente, max_passos=100):
    """Loop percepção → decisão → ação.
    
    Esta função aparece em TODA aula da Semana 2 à 16.
    O que muda é apenas a complexidade de agente.decidir().
    """
    obs = ambiente.reset()
    recompensa_total = 0
    done = False
    passo = 0
    
    while not done and passo < max_passos:
        acao = agente.decidir(obs)           # ← decide
        obs, recompensa, done = ambiente.step(acao)  # ← age
        recompensa_total += recompensa
        passo += 1
    
    return recompensa_total, passo

print("Loop percepção → decisão → ação")
print("Usaremos esta estrutura em todas as semanas.")

## 4. Resumo da Semana 1

| Conceito | O que você aprendeu |
|---|---|
| **Agente racional** | Sistema que percebe e age para maximizar uma métrica de desempenho |
| **4 visões da IA** | Pensar/Agir × Humana/Racional — o curso foca em "agir racionalmente" |
| **IA Simbólica vs. ML** | Programar regras vs. aprender de dados — a Bitter Lesson explica por que a segunda vence |
| **Loop universal** | `obs → decidir(obs) → ação` — a interface que unifica o curso |
| **Bitter Lesson** | Métodos gerais + computação > conhecimento artesanal (Sutton, 2019) |

**Pergunta condutora respondida:** A IA funciona agora porque, depois de 70 anos tentando programar inteligência à mão, o campo aprendeu que métodos gerais que aprendem de dados e escalam com computação vencem.

---

*Prof. Marcelo Keese Albertini · FACOM · UFU · GBC063 · Semana 1*